In [ ]:
# ============================================
# DEEPFAKE DETECTION - MULTI-DATASET TRAINING
# ============================================
# Training on combined DFDC02 + DFD01 datasets.
# 4 ablation experiments: full, spatial, temporal, sequential.
# Handles DFD01 class imbalance (363 real vs 3068 fake).
#
# PREREQUISITES:
# - Kaggle dataset 1: preprocessed_DFDC02_16 (real/fake face crops)
# - Kaggle dataset 2: preprocessed_DFD01_16 (real/fake face crops)
# Both must be added to the notebook as input datasets.
# GPU accelerator must be enabled (T4 or P100).

import subprocess, sys, os

# Step 1: Install dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch'],
               check=True)
print('Dependencies installed.')

# Step 2: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
    print('Project cloned.')
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'],
                   check=True)
    print('Project updated.')

# Step 3: Write multi-dataset training script
train_script = r'''
import os, sys, time, json, gc
os.chdir('/kaggle/working/project')
sys.path.insert(0, '/kaggle/working/project')

import numpy as np
print(f"numpy version: {np.__version__}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU!'}")
assert torch.cuda.is_available(), "No GPU detected!"
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU Memory: {mem_gb:.1f} GB")

# =============================================
# Find BOTH preprocessed datasets
# =============================================
print("\n" + "="*60)
print("SEARCHING FOR PREPROCESSED DATASETS")
print("="*60)

def find_preprocessed(keyword):
    """Find preprocessed dataset (has real/ and fake/ subdirs with image folders)."""
    candidates = []
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'real' in dirs and 'fake' in dirs:
            real_p = os.path.join(root, 'real')
            fake_p = os.path.join(root, 'fake')
            rc = len([d for d in os.listdir(real_p) if os.path.isdir(os.path.join(real_p, d))])
            fc = len([d for d in os.listdir(fake_p) if os.path.isdir(os.path.join(fake_p, d))])
            if rc > 0 and fc > 0 and keyword.lower() in root.lower():
                candidates.append((root, rc, fc))
    # Return deepest match (most specific path)
    if candidates:
        candidates.sort(key=lambda x: len(x[0]), reverse=True)
        return candidates[0]
    return None, 0, 0

dfdc02_path, dfdc02_real, dfdc02_fake = find_preprocessed('dfdc02')
dfd01_path, dfd01_real, dfd01_fake = find_preprocessed('dfd01')

# Fallback: scan ALL datasets with real/fake structure
if dfdc02_path is None or dfd01_path is None:
    print("Keyword search incomplete, scanning all directories...")
    all_datasets = []
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'real' in dirs and 'fake' in dirs:
            real_p = os.path.join(root, 'real')
            fake_p = os.path.join(root, 'fake')
            rc = len([d for d in os.listdir(real_p) if os.path.isdir(os.path.join(real_p, d))])
            fc = len([d for d in os.listdir(fake_p) if os.path.isdir(os.path.join(fake_p, d))])
            if rc > 0 and fc > 0:
                all_datasets.append((root, rc, fc))
                print(f"  Found: {root} (real={rc}, fake={fc})")

    if len(all_datasets) < 2:
        print("\nERROR: Need 2 preprocessed datasets but found fewer!")
        print("Make sure both preprocessed_DFDC02_16 and preprocessed_DFD01_16")
        print("are added as input datasets in this notebook.")
        print("\nListing /kaggle/input/:")
        for root, dirs, files in os.walk('/kaggle/input'):
            level = root.replace('/kaggle/input', '').count(os.sep)
            if level < 3:
                indent = '  ' * level
                print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)')
        sys.exit(1)

    # Auto-identify: DFDC02 is more balanced, DFD01 is heavily fake-biased
    for path, rc, fc in all_datasets:
        ratio = fc / max(rc, 1)
        if ratio < 3.0 and dfdc02_path is None:
            dfdc02_path, dfdc02_real, dfdc02_fake = path, rc, fc
        elif ratio >= 3.0 and dfd01_path is None:
            dfd01_path, dfd01_real, dfd01_fake = path, rc, fc

if dfdc02_path is None or dfd01_path is None:
    print(f"ERROR: Could not identify both datasets!")
    print(f"  DFDC02: {dfdc02_path}")
    print(f"  DFD01:  {dfd01_path}")
    sys.exit(1)

print(f"\n[OK] DFDC02: {dfdc02_path}")
print(f"     Real: {dfdc02_real}, Fake: {dfdc02_fake}, Total: {dfdc02_real + dfdc02_fake}")
print(f"     Ratio fake/real: {dfdc02_fake / max(dfdc02_real, 1):.2f}")

print(f"\n[OK] DFD01: {dfd01_path}")
print(f"     Real: {dfd01_real}, Fake: {dfd01_fake}, Total: {dfd01_real + dfd01_fake}")
print(f"     Ratio fake/real: {dfd01_fake / max(dfd01_real, 1):.2f}")

total_real = dfdc02_real + dfd01_real
total_fake = dfdc02_fake + dfd01_fake
total_all = total_real + total_fake
print(f"\n{'='*60}")
print(f"COMBINED: {total_all} videos (Real={total_real}, Fake={total_fake})")
print(f"Combined ratio fake/real: {total_fake / max(total_real, 1):.2f}")
print(f"{'='*60}")

# =============================================
# Run 4 ablation experiments
# =============================================
from config import Config
from train import train
from utils import save_metrics

combined_dataset_root = f"{dfdc02_path}+{dfd01_path}"
combined_dataset_name = "dfdc02+dfd01"
SPLIT_FILENAME = "split_multi_seed42.json"

# Class imbalance handling
use_sampler = abs(total_fake / max(total_real, 1) - 1.0) > 0.5
pos_weight_value = max(0.33, min(3.0, total_real / max(total_fake, 1)))
print(f"\nClass imbalance strategy:")
print(f"  WeightedRandomSampler: {use_sampler}")
print(f"  pos_weight: {pos_weight_value:.4f}")

EXPERIMENTS = [
    {'name': 'A1_full_model',    'model_type': 'full',       'fusion_type': 'adaptive'},
    {'name': 'A2_spatial_only',  'model_type': 'spatial',    'fusion_type': 'adaptive'},
    {'name': 'A3_temporal_only', 'model_type': 'temporal',   'fusion_type': 'adaptive'},
    {'name': 'A4_sequential',    'model_type': 'sequential', 'fusion_type': 'adaptive'},
]

all_results = []
total_start = time.time()

for i, exp in enumerate(EXPERIMENTS, 1):
    print(f"\n{'='*70}")
    print(f"EXPERIMENT [{i}/4]: {exp['name']}")
    print(f"  model_type={exp['model_type']}, fusion_type={exp['fusion_type']}")
    print(f"  dataset: {combined_dataset_name} ({total_all} videos)")
    print(f"{'='*70}")

    cfg = Config()
    cfg.dataset_root = combined_dataset_root
    cfg.dataset_name = combined_dataset_name
    cfg.model_type = exp['model_type']
    cfg.fusion_type = exp['fusion_type']

    # Training hyperparameters
    cfg.max_epochs = 30
    cfg.batch_size = 16
    cfg.patience = 7
    cfg.device = 'auto'
    cfg.use_amp = True
    cfg.num_workers = 2
    cfg.pin_memory = True
    cfg.seed = 42
    cfg.unfreeze_last_n_blocks = 2

    # Class imbalance
    if use_sampler:
        cfg.use_weighted_sampler = True
        cfg.use_class_weights = False
    else:
        cfg.use_weighted_sampler = False
        if abs(pos_weight_value - 1.0) > 0.2:
            cfg.pos_weight = round(pos_weight_value, 4)

    # Split: first creates, rest reuse
    cfg.splits_dir = './splits'
    cfg.split_filename = SPLIT_FILENAME
    cfg.output_dir = './experiments'

    if i == 1:
        cfg.split_mode = 'random'
        cfg.save_split = True
        print(f"  Split: creating new ({SPLIT_FILENAME})")
    else:
        cfg.split_mode = 'fixed'
        cfg.save_split = False
        print(f"  Split: reusing ({SPLIT_FILENAME})")

    t0 = time.time()
    try:
        metrics = train(cfg)
        duration = round((time.time() - t0) / 60, 1)
        metrics['experiment_name'] = exp['name']
        metrics['status'] = 'success'
        metrics['duration_min'] = duration
        metrics['training_datasets'] = ['dfdc02', 'dfd01']
        all_results.append(metrics)

        test = metrics.get('test', {})
        print(f"\n{'*'*50}")
        print(f"[OK] {exp['name']} COMPLETED in {duration} min")
        print(f"     Test AUC  = {test.get('auc', 0):.4f}")
        print(f"     Test Acc  = {test.get('accuracy', 0):.4f}")
        print(f"     Test F1   = {test.get('f1', 0):.4f}")
        print(f"     Test EER  = {test.get('eer', 0):.4f}")
        print(f"     Best Epoch= {metrics.get('best_epoch', '?')}")
        print(f"{'*'*50}")
    except Exception as e:
        duration = round((time.time() - t0) / 60, 1)
        print(f"\n[FAIL] {exp['name']} after {duration} min")
        print(f"Error: {e}")
        import traceback; traceback.print_exc()
        all_results.append({
            'experiment_name': exp['name'],
            'status': 'failed',
            'error': str(e),
            'duration_min': duration,
        })

    # Cleanup GPU memory between experiments
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    alloc = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU Memory: {alloc:.2f} GB allocated, {reserved:.2f} GB reserved")

    # Save intermediate results
    save_metrics(all_results, './experiments/all_results_multi.json')

# =============================================
# Final Summary
# =============================================
total_time = round((time.time() - total_start) / 60, 1)

print(f"\n\n{'='*80}")
print(f"MULTI-DATASET TRAINING — FINAL RESULTS")
print(f"Training data: DFDC02 ({dfdc02_real+dfdc02_fake}) + DFD01 ({dfd01_real+dfd01_fake}) = {total_all} videos")
print(f"Total training time: {total_time} min ({total_time/60:.1f} hours)")
print(f"{'='*80}")
print(f"{'Experiment':<25} {'AUC':>8} {'Acc':>8} {'F1':>8} {'EER':>8} {'Epoch':>6} {'Time':>8}")
print('-' * 80)

successful = 0
for r in all_results:
    if r.get('status') == 'success':
        successful += 1
        t = r.get('test', {})
        be = r.get('best_epoch', '?')
        print(f"{r['experiment_name']:<25} {t.get('auc',0):>8.4f} {t.get('accuracy',0):>8.4f} "
              f"{t.get('f1',0):>8.4f} {t.get('eer',0):>8.4f} {be:>6} {r.get('duration_min',0):>7.1f}m")
    else:
        print(f"{r['experiment_name']:<25} {'FAILED':>8}  {r.get('error','')[:45]}")

print(f"\nSuccessful: {successful}/4")

# Find best model
best = max([r for r in all_results if r.get('status') == 'success'],
           key=lambda r: r.get('test', {}).get('auc', 0), default=None)
if best:
    bt = best.get('test', {})
    print(f"\nBEST MODEL: {best['experiment_name']}")
    print(f"  AUC={bt.get('auc',0):.4f}, Acc={bt.get('accuracy',0):.4f}, F1={bt.get('f1',0):.4f}")

# Copy results for download
os.makedirs('/kaggle/working/experiments', exist_ok=True)
os.system('cp -r experiments/ /kaggle/working/experiments/ 2>/dev/null')
os.system('cp -r splits/ /kaggle/working/splits/ 2>/dev/null')
os.system('tar czf /kaggle/working/results_multi.tar.gz -C /kaggle/working/project experiments/ splits/ 2>/dev/null')
print(f"\nresults_multi.tar.gz ready for download")
'''

with open('/kaggle/working/run_multi_training.py', 'w') as f:
    f.write(train_script)

# Step 4: Run training in subprocess with REAL-TIME output
print('\n=== Starting multi-dataset training ===')
print('Output will appear in real-time below:\n')

proc = subprocess.Popen(
    [sys.executable, '-u', '/kaggle/working/run_multi_training.py'],
    cwd='/kaggle/working/project',
    env={**os.environ, 'PYTHONPATH': '/kaggle/working/project'},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nTraining subprocess exited with code: {proc.returncode}')

# Copy results
if os.path.exists('/kaggle/working/project/experiments'):
    subprocess.run(['cp', '-r', '/kaggle/working/project/experiments/', '/kaggle/working/experiments/'])
    print('Results copied to /kaggle/working/experiments/')
if os.path.exists('/kaggle/working/project/splits'):
    subprocess.run(['cp', '-r', '/kaggle/working/project/splits/', '/kaggle/working/splits/'])
    print('Splits copied to /kaggle/working/splits/')
if os.path.exists('/kaggle/working/project/results_multi.tar.gz'):
    subprocess.run(['cp', '/kaggle/working/project/results_multi.tar.gz', '/kaggle/working/'])
    print('results_multi.tar.gz ready for download')